In [1]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path

In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

In [5]:
DATASET_DIR = PROJECT_DIR / "ESC-50"
AUDIO_DIR = DATASET_DIR / "audio"
METADATA_FILE = DATASET_DIR / "meta" / "esc50.csv"

df = pd.read_csv(METADATA_FILE)

print("Recordings:", len(df))
print("Classes:", df["category"].nunique())
print("Folds:", sorted(df["fold"].unique()))

Recordings: 2000
Classes: 50
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [7]:
SAMPLE_RATES = 22050
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

In [8]:
def audio_logs_extractor(audiopath):
    audio, sr = librosa.load(
        audiopath,
        sr=SAMPLE_RATES,
        mono=True
    )

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel


example_path = AUDIO_DIR / df.iloc[0]["filename"]
example_features = audio_logs_extractor(example_path)

In [9]:
print("Feature shape:", example_features.shape)
print("Category:", df.iloc[0]["category"])
print("All values finite:", np.isfinite(example_features).all())

Feature shape: (128, 216)
Category: dog
All values finite: True


In [10]:
from tqdm.auto import tqdm

features = []
labels = []
folds = []

for row in tqdm(df.itertuples(index=False), total=len(df)):
    audio_path = AUDIO_DIR / row.filename

    log_mel = audio_logs_extractor(audio_path)

    features.append(log_mel)
    labels.append(row.target)
    folds.append(row.fold)

X = np.stack(features).astype(np.float32)
y = np.array(labels, dtype=np.int64)
folds = np.array(folds, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fold shape:", folds.shape)
print("All features finite:", np.isfinite(X).all())

/Volumes/Coding Work Place/Deep learning Envirnoment Audio Detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2000/2000 [00:24<00:00, 80.76it/s] 


X shape: (2000, 128, 216)
y shape: (2000,)
Fold shape: (2000,)
All features finite: True


In [11]:
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUTPUT_DIR / "esc50_features.npz",
    X=X,
    y=y,
    folds=folds
)

print("Saved:", OUTPUT_DIR / "esc50_features.npz")

Saved: /Volumes/Coding Work Place/Deep learning Envirnoment Audio Detection/outputs/esc50_features.npz


In [12]:
print(X.shape)
print(y.shape)
print(np.unique(y).size)
print(np.unique(folds, return_counts=True))

(2000, 128, 216)
(2000,)
50
(array([1, 2, 3, 4, 5]), array([400, 400, 400, 400, 400]))
